## Transition Recursive equation of Local Linear SDE

From locally linear approximation method, we evaluated the drift coefficient $A$ and the control vector $\alpha{}$ using neural net.

The key strength of linear approximation is that the SDE could be solved analytically without simulation - namely, simulation-free property.

Therefore, the average and covariance of the normal distribution ( local linear approxmaiton is a type of OU process, and the closed solution distribution is given; $X_{t} \sim \mathcal{N}(\mu{}_t,\Sigma{}_t)$ is calculated directly from the transition recursive equation.

Simulation-free property gives rise to the independent gaussian sampling at each time stamp of the given process.




### linear SDE coefficient

In [3]:
import torch
from torch import nn

from src import make_ou_data
from src import ObsEncoder
from src import ACSSMTransformer, masking_unseen_data_Full_Assim
from src import PSDLocalLinearApprox

# 1. Data preparation
data = make_ou_data(32, 40, 3)
obs = data["observation"]
obs_mask = data["obs_mask"]
times = data["times"]

# 2. Encoding the observation
encoder = ObsEncoder(obs_dim= obs.shape[-1], embed_dim=8)
sampled_latent_obs = encoder(obs, obs_mask)
B,T,D =sampled_latent_obs.shape # (32, 40, 8)

# 2.1 Transformer layer
transformer = ACSSMTransformer(D,2)
z_raw = transformer(sampled_latent_obs, times, obs_mask)
z = masking_unseen_data_Full_Assim(z_raw, obs_mask)

# 3. Local linear approximation
localLinApp = PSDLocalLinearApprox(D,D)
linear_SDE_param = localLinApp(z)




time_embedding: torch.Size([32, 40, 8])
transformer_input: torch.Size([32, 40, 8])
z_raw: torch.Size([32, 40, 8])


In [ ]:

dt = times[:, 1:] - times[:, :-1]

# exclude the final value of A and alpha.
# at each interval, use the value at initial time.
lam = linear_SDE_param["eigen_vals"][:, :-1,:]
alpha = linear_SDE_param["alpha"][:,:-1,:]


print(lam.shape, alpha.shape, dt.shape)

torch.Size([32, 39, 8]) torch.Size([32, 39, 8]) torch.Size([32, 39])


### Change basis and one-step integration

The original LSDE is
$$
    dX_t = (-A_iX_t + \alpha{})dt + dW_t
$$

change basis using $E$

$$
    \hat{X}_t = E^{\intercal}X_t, \quad{} \hat{\alpha{}}_i = E^{\intercal}\alpha{}_i, \quad{} A_i \to D_i=\text{diag}(\lambda{}_i)
$$

Therfore, SDE turns into simple 1-d SDE
$$
    d\hat{X}_{t} = \left( -\text{diag}(\lambda{}_i)\hat{X}_t + \hat{\alpha{}}_i \right)dt + d\hat{W}_t
$$

We can integrate out a time step to get transition equation

$$
    \hat{X}_{t+\Delta{}} = e^{-\lambda{}\Delta{}}\hat{X}_t 
    + \hat{\alpha{}}\int_{0}^{\Delta{}} e^{-\lambda{}u}\,\mathrm{d}u 
    + \int_{t}^{t+\Delta{}} e^{-\lambda{}(t+\Delta{}-s)}\,\mathrm{d}W_s 
$$

We define 3 coefficient here

$$
    f = e^{-\lambda{}\Delta{}}, \quad{} g = \int_{0}^{\Delta{}} e^{-\lambda{}u}\,\mathrm{d}u = \frac{1-e^{-\lambda{}\Delta{}}}{\lambda{}}  
$$





In [5]:
def phi(x: torch.Tensor) -> torch.Tensor:
    """
    phi(x) = (1 - exp(-x)) / x
    phi(0) = 1

    Input / output: same shape
    """
    small = x.abs() < 1e-4

    # 일반식 branch에서도 0으로 나누지 않도록 한다.
    x_regular = torch.where(
        small,
        torch.ones_like(x),
        x,
    )
    regular = -torch.expm1(-x_regular) / x_regular

    # Taylor식은 작은 x에서만 평가하도록 입력을 제한한다.
    x_small = torch.where(
        small,
        x,
        torch.zeros_like(x),
    )
    series = (
        1.0
        - x_small / 2.0
        + x_small.square() / 6.0
        - x_small**3 / 24.0
    )

    return torch.where(small, series, regular)

In [6]:
def transition_coefficients(
    local_params: dict[str, torch.Tensor],
    times: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """
    Convention:
        parameters at index i apply on [times[i], times[i+1])

    Assumes:
        dX = (-A X + alpha) dt + dW
        A_i = E diag(lambda_i) E.T
        E is shared across all batches and intervals.

    All returned tensors:
        (B, T-1, d)
    """
    eigen_vals = local_params["eigen_vals"]  # (B, T, d)
    alpha_all = local_params["alpha"]       # (B, T, d)
    E = local_params["E"]                   # (d, d)

    times = times.to(
        device=eigen_vals.device,
        dtype=eigen_vals.dtype,
    )

    
    """
        (t_1,t_2, ..., t_T-1)
       -(t_0,t_1, ..., t_T-2)
       =(dt0,dt1, ..., dt_T-2)

        dt_i = [ t_i, t_i+1 )
        A_i,alpha_i  <->  dt_i

        => Total T-1 intervals
    """
    lam = eigen_vals[:, :-1, :]             # (B, T-1, d)
    alpha = alpha_all[:, :-1, :]            # (B, T-1, d)

    dt = (
        times[:, 1:] - times[:, :-1]
    ).unsqueeze(-1)                         # (B, T-1, 1)

    # 2. Control을 공통 고유기저로 옮긴다.
    alpha_hat = alpha @ E                   # (B, T-1, d)

    # 3. 각 고유방향의 전이 계수를 계산한다.
    x = lam * dt                            # (B, T-1, d)

    decay = torch.exp(-x)
    control_gain = dt * phi(x)
    noise_var = dt * phi(2.0 * x)

    # 4. 실제 affine 이동량과 분산 감쇠 계수.
    shift_hat = control_gain * alpha_hat
    var_decay = decay.square()

    return {
        "decay": decay,
        "control_gain": control_gain,
        "shift_hat": shift_hat,
        "noise_var": noise_var,
        "var_decay": var_decay,
    }